# H1 V2 · Counts as text for the VLM (KR2, all six tasks re-scored)

Task **H1**, variant **V2**, from `Essential_task_delegation/Next_Steps_Roadmap.md`. Goal **O2 / KR2**: re-score all six DisasterM3 tasks with the "textual bridge", a U-Net fact line in front of each question.

**What it does**
- **Model:** our fine-tuned Qwen2.5-VL-7B (`AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP`), fp16, split across the two T4s with Hugging Face transformers.
  - It doesn't use vLLM: current vLLM wheels need CUDA 13 and torch 2.13, while Kaggle has torch 2.11 + CUDA 12.8.
  - The 499A vLLM run (`3b_hybrid_vllm_evaluation.ipynb`, 1 Aug) crashed on exactly that: `libnvrtc.so.13`.
- **Prompts:** the official DisasterM3 templates (`temp_junjue/pyscripts/run_vllm.py`), with images resized to 512 × 512 as in the 499A evaluation.
- **Two variants, same model, same settings, greedy decoding:**
  - **plain:** the official prompt;
  - **facts:** the same prompt with V1's fact line before "Question:", for example "U-Net analysis: 51 intact, 3 damaged and 0 destroyed buildings; most damage in the north-east of the image."
- **Tasks:**
  - Disaster type (DTR), bearing bodies (BBR), road damage (DRE) and land use (DSR) run in both variants.
  - Relational reasoning (ORR) runs plain only: its images are separate crops with no U-Net facts.
  - Building damage counting (BDC) isn't asked to the VLM: V1's counting answers fill that row.
- **Scoring:** the 499A scorer's rule, where the set of option letters must match exactly.

**Why greedy, and why both variants in one run**
- The 499A runner was patched to skip the model's generation settings, so it most likely sampled at temperature 1.0.
- Re-running that setup on 1 Aug gave Disaster Type 64.05% instead of 68.33%, so the 499A numbers move by several points between runs.
- So the facts run is compared with this notebook's own plain run. The 499A numbers stay in the table for reference.

**How to run on Kaggle**
1. Settings: **GPU T4 ×2**, internet ON.
2. Inputs (found by file name):
   - `redwannewaz/disasterm3-bench-mirror-v2` (benchmark JSON and images);
   - `h1-v1-facts`, the output of the **re-run** H1 V1 (`facts.json` and `H1_v1_summary.csv`). The first V1 run's `facts.json` is empty, and this notebook stops on it.
3. Secret `HF_TOKEN` (AbrarAlam), in case the model repo is private.
4. Run with `SMOKE_TEST = True` first: 10 questions per task and variant, about 15 minutes including the 16 GB model download. Then use Save & Run All, about 3–4 hours for roughly 15,000 answers.
   - Answers are written as they come, and a re-run skips them.
   - The run stops cleanly at 11 hours. To continue, attach that version's output and run again.
5. Outputs in `/kaggle/working/H1V2/`:
   - `kr2_results.csv` (the six-task table), `H1_v2_per_question.csv`, `config.json`, `README.md`;
   - `responses/<variant>/<subset>.jsonl`, and `results/<subset>/<tag>/finished.jsonl` in the format `4_score_results.py` reads.

**Gate:** keep the facts only if no task drops by more than 1 point against the plain run. The counting half of KR2 is Gate G4, from V1.

In [1]:
# ── Cell 1: Environment (Kaggle's own torch; transformers only if too old for Qwen2.5-VL) ──
import importlib.metadata as _md
import subprocess
import sys

def _version(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return "0"

def _older(have, want):
    nums = lambda v: tuple(int(x) for x in v.split(".")[:2] if x.isdigit())
    return nums(have) < nums(want)

if _older(_version("transformers"), "4.49") or _version("accelerate") == "0":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.49", "accelerate"], check=True)

import torch
import transformers
print(f"✓ torch {torch.__version__}, transformers {transformers.__version__}, python {sys.version.split()[0]}")
print(f"  GPUs: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
assert torch.cuda.device_count() >= 2, "Pick GPU T4 x2: the 7B model in fp16 needs both T4s"

✓ torch 2.11.0+cu128, transformers 5.16.1, python 3.13.15
  GPUs: ['Tesla T4', 'Tesla T4']


In [2]:
# ── Cell 2: Configuration, the benchmark, V1's facts and the images ────────
import glob
import json
import os
import re
import shutil
import time
from collections import Counter
from pathlib import Path

import pandas as pd

MODEL_ID = "AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP"   # our fine-tuned 7B (CSE499A), merged
OUT_DIR = Path("/kaggle/working/H1V2")
SMOKE_TEST = False             # True = 10 questions per task and variant
IMAGE_SIZE = 512               # images resized to 512 x 512, as in the 499A evaluation
BATCH = 8                      # questions per generate() call; halved automatically on out-of-memory
MAX_NEW_TOKENS = 32            # answers are option letters, e.g. "B, E, G"
TIME_LIMIT_H = 11.0            # stop cleanly before Kaggle's 12-hour limit
# (code, subset name used by the 499A scorer, task name in the benchmark, gets V1's fact line?)
TASKS = [("DTR", "disaster_type", "Disaster Type Recognition", True),
         ("BBR", "bearing_body", "Disaster Bearing Bodies Recognition", True),
         ("DRE", "road_damage_counting", "Road Damage Counting", True),
         ("DSR", "landuse", "Disaster Scene Recognition", True),
         ("ORR", "relational_reasoning_qa", "Relational Reasoning", False)]   # separate crop images: no facts
VARIANTS = ["plain", "facts"]
REF_FT = {"DTR": 68.33, "BBR": 16.39, "BDC": 23.58, "DRE": 26.24, "DSR": 31.31, "ORR": 22.50}   # 499A fine-tuned 7B
REF_BASE = {"DTR": 66.6, "BBR": 4.7, "BDC": 34.2, "DRE": 29.3, "DSR": 28.3, "ORR": 23.9}       # base Qwen2.5-VL-7B

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

def find_one(pattern, why):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"No {pattern} under /kaggle/input: {why}")
    return Path(hits[0])

def name_of(path):
    return Path(str(path).replace("\\", "/")).name

with open(find_one("benchmark_release.json", "attach redwannewaz/disasterm3-bench-mirror-v2"), encoding="utf-8") as f:
    bench = json.load(f)
with open(find_one("facts.json", "attach h1-v1-facts (the output of the re-run H1 V1)"), encoding="utf-8") as f:
    FACTS = json.load(f)
v1_hits = sorted(glob.glob("/kaggle/input/**/H1_v1_summary.csv", recursive=True))
V1_SUMMARY = pd.read_csv(v1_hits[0]) if v1_hits else None

bench_index = {}                # every benchmark image by file name: mirrors nest them differently
for p in glob.glob("/kaggle/input/**/test_images/**/*.png", recursive=True):
    bench_index.setdefault(Path(p).name, p)

questions = {}                  # code -> [(id as in the 499A scorer, entry)]
for code, subset, task, _ in TASKS:
    qs = [(f"{subset}_{i}", e) for i, e in enumerate(e for e in bench if e.get("task") == task)]
    questions[code] = qs[:10] if SMOKE_TEST else qs

needed = {name_of(e[k]) for qs in questions.values() for _, e in qs
          for k in ("pre_image_path", "post_image_path", "image_path") if e.get(k)}
found = sum(n in bench_index for n in needed)
fact_names = {name_of(e["post_image_path"]).replace("_sar", "") for code, _, _, with_facts in TASKS if with_facts
              for _, e in questions[code]}
with_fact = sum(n in FACTS for n in fact_names)
if found < 0.99 * len(needed):
    raise FileNotFoundError(f"Only {found:,} of {len(needed):,} benchmark images found: attach the bench mirror.")
if with_fact < 0.99 * len(fact_names):
    raise ValueError(f"facts.json covers {with_fact:,} of {len(fact_names):,} after-images: re-run H1 V1 "
                     "(its first run found no benchmark images and wrote an empty facts.json).")

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"✓ H1 V2{'  [SMOKE TEST]' if SMOKE_TEST else ''}: {MODEL_ID}")
print(f"  Questions: {({c: len(q) for c, q in questions.items()})}")
print(f"  Images found: {found:,} of {len(needed):,}; V1 facts for {with_fact:,} of {len(fact_names):,} after-images")
print(f"  V1 summary for the BDC row: {'found' if V1_SUMMARY is not None else 'not attached'}")

✓ H1 V2: AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP
  Questions: {'DTR': 420, 'BBR': 2363, 'DRE': 2178, 'DSR': 2007, 'ORR': 1018}
  Images found: 5,694 of 5,694; V1 facts for 2,149 of 2,150 after-images
  V1 summary for the BDC row: found


In [3]:
# ── Cell 3: The official DisasterM3 prompts, the fact line, and the 499A scorer ──
from PIL import Image

# Copied from temp_junjue/pyscripts/run_vllm.py (the DisasterM3 authors' benchmark runner)
PROMPTS = dict(
    bearing_body="""Analyze both the pre-disaster and post-disaster images to answer the following question. Choose the best option(s) from the candidate options provided.

pre-disaster image:
<image>

post-disaster image:
<image>

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letters of the correct options, separated by a comma and a space (e.g., C, D, H). Do not include any explanation or other text.""",
    landuse="""Analyze the image to answer the following question. Choose the best option(s) from the candidate options provided.

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letters of the correct options, separated by a comma and a space (e.g., C, D, H). Do not include any explanation or other text.""",
    single_choice="""Analyze both the pre-disaster and post-disaster images to answer the following question. Choose the best option from the candidate options provided.

pre-disaster image:
<image>

post-disaster image:
<image>

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text.""",
    relational_reasoning_qa="""Analyze the image to answer the following question. Choose the best option from the candidate options provided.

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text.""",
)

def fact_for(e, use_facts):
    """V1's fact line for the question's after-image (SAR questions use the optical twin), or None."""
    if not use_facts:
        return None
    f = FACTS.get(name_of(e.get("post_image_path", "")).replace("_sar", ""))
    return f["text"] if f else None

def build(e, subset, fact):
    """-> (chat messages, PIL images in prompt order), built as the official runner builds them."""
    if subset == "landuse":
        text, paths = PROMPTS["landuse"].format(prompt=e["prompts"], options_str=e["options_str"]), [e["pre_image_path"]]
    elif subset == "relational_reasoning_qa":     # this task's entries spell it option_str
        text, paths = PROMPTS[subset].format(prompt=e["prompts"], options_str=e["option_str"]), [e["image_path"]]
    else:
        key = "bearing_body" if subset == "bearing_body" else "single_choice"
        text = PROMPTS[key].format(prompt=e["prompts"], options_str=e["options_str"])
        paths = [e["pre_image_path"], e["post_image_path"]]
    if fact:
        text = text.replace("Question: ", f"{fact}\n\nQuestion: ", 1)
    parts = text.split("<image>")
    if len(parts) == 1:                            # single-image templates: the image comes first
        content = [{"type": "image"}, {"type": "text", "text": text}]
    else:
        content = [{"type": "text", "text": parts[0]}, {"type": "image"}, {"type": "text", "text": parts[1]},
                   {"type": "image"}, {"type": "text", "text": parts[2]}]
    images = [Image.open(bench_index[name_of(p)]).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.BICUBIC)
              for p in paths]
    return [{"role": "user", "content": content}], images

def letters(text):
    """The 499A scorer (4_score_results.py): the set of capital letters left after removing
    'THE ANSWER IS' and 'OPTION'. An answer counts only if this set equals the truth's set."""
    t = str(text).upper().replace("THE ANSWER IS", "").replace("OPTION", "")
    return set(re.findall(r"[A-Z]", t))

# Self-checks: the scorer, and where the fact line lands
assert letters("B, E, G") == {"B", "E", "G"} and letters("The answer is D.") == {"D"} and letters("") == set()
_e = questions["BBR"][0][1]
_msgs, _imgs = build(_e, "bearing_body", "U-Net analysis: TEST.")
_text = "".join(c.get("text", "") for c in _msgs[0]["content"])
assert len(_imgs) == 2 and "U-Net analysis: TEST.\n\nQuestion:" in _text and _text.count("U-Net") == 1
assert len(build(questions["ORR"][0][1], "relational_reasoning_qa", None)[1]) == 1
print("✓ Prompts ready (self-checks passed). Example with a fact line:\n")
print(_text[_text.index("U-Net"):][:400])

✓ Prompts ready (self-checks passed). Example with a fact line:

U-Net analysis: TEST.

Question: What essential land-cover objects appear damaged in this disaster zone?
Options: A. Stadiums, B. Buildings, C. Coastline, D. Dams, E. Open-space ground, F. Forest, G. Roads, H. Storage tank.

Your task is to respond with ONLY the capital letters of the correct options, separated by a comma and a space (e.g., C, D, H). Do not include any explanation or other text.


In [4]:
# ── Cell 4: Load the model (fp16, split across both T4s) and the batched greedy generator ──
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID, token=hf_token())
processor.tokenizer.padding_side = "left"              # batched generation needs left padding
# transformers 4.56 renamed torch_dtype to dtype; older versions may ignore dtype and load fp32 (30 GB)
dtype_arg = "torch_dtype" if _older(transformers.__version__, "4.56") else "dtype"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, **{dtype_arg: torch.float16}, device_map="auto", attn_implementation="sdpa", token=hf_token())
model.eval()

@torch.inference_mode()
def generate(batch):
    """batch: list of (messages, images) -> list of answers. Greedy, so a re-run gives the same answers."""
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m, _ in batch]
    images = [img for _, imgs in batch for img in imgs]
    inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=None,
                         top_p=None, top_k=None, repetition_penalty=1.0)
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    return [s.strip() for s in processor.batch_decode(new_tokens, skip_special_tokens=True)]

def generate_safe(batch):
    """Halves the batch on out-of-memory instead of crashing the run."""
    try:
        return generate(batch)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(batch) == 1:
            return [""]
        half = len(batch) // 2
        return generate_safe(batch[:half]) + generate_safe(batch[half:])

# One question of each prompt type, plain and with facts, before the long run
for code, subset, _, with_facts in TASKS:
    e = questions[code][0][1]
    answers = generate_safe([build(e, subset, None)] + ([build(e, subset, fact_for(e, True))] if with_facts else []))
    print(f"  {code}: truth {e['ground_truth_option']!r} | plain {answers[0]!r}"
          + (f" | facts {answers[1]!r}" if with_facts else ""))
print(f"✓ Model loaded ({dtype_arg}=float16) and checked in {(time.time() - t0) / 60:.1f} min; "
      f"GPU memory {[f'{torch.cuda.memory_allocated(i) / 1e9:.1f} GB' for i in range(torch.cuda.device_count())]}")

preprocessor_config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.78k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

  DTR: truth 'E' | plain 'A' | facts 'A'
  BBR: truth 'B, E, F, G.' | plain 'B, E, F, G' | facts 'B, E, F, G'
  DRE: truth 'C' | plain 'B' | facts 'B'
  DSR: truth 'C, D, H.' | plain 'D, H' | facts 'D, H'
  ORR: truth 'A' | plain 'B'
✓ Model loaded (dtype=float16) and checked in 2.4 min; GPU memory ['7.6 GB', '9.0 GB']


## Run: every question, plain and then with facts
Tasks run in the order DTR, BBR, DRE, DSR, ORR, each one plain and then with facts. So if the 11-hour limit stops the run, the finished tasks have complete pairs. Answers are appended to `responses/<variant>/<subset>.jsonl` after every batch; a re-run (or a new version with this one's output attached) skips the questions already answered.

In [5]:
# ── Cell 5: Ask every question (plain, then with facts), writing answers as they come ──
from tqdm.auto import tqdm

# Continue an earlier run: copy its answers in when that run's output is attached
previous = sorted(glob.glob("/kaggle/input/**/H1V2/responses", recursive=True))
if previous and not (OUT_DIR / "responses").exists():
    shutil.copytree(previous[0], OUT_DIR / "responses")
    print(f"Continuing from {previous[0]}")

t_start = time.time()
stopped, unreadable = False, Counter()
for code, subset, _, with_facts in TASKS:
    for variant in VARIANTS:
        if variant == "facts" and not with_facts:
            continue
        path = OUT_DIR / "responses" / variant / f"{subset}.jsonl"
        path.parent.mkdir(parents=True, exist_ok=True)
        done = {json.loads(line)["id"] for line in open(path, encoding="utf-8")} if path.exists() else set()
        todo = [(qid, e) for qid, e in questions[code] if qid not in done]
        with open(path, "a", encoding="utf-8") as f:
            for i in tqdm(range(0, len(todo), BATCH), desc=f"{code} {variant}"):
                if (time.time() - t_start) / 3600 > TIME_LIMIT_H:
                    stopped = True
                    break
                built = []
                for qid, e in todo[i:i + BATCH]:
                    try:
                        built.append((qid, build(e, subset, fact_for(e, variant == "facts"))))
                    except (KeyError, OSError):              # image missing or unreadable: no answer, scored wrong
                        unreadable[code] += 1
                        f.write(json.dumps({"id": qid, "response": ""}) + "\n")
                answers = generate_safe([b for _, b in built]) if built else []
                for (qid, _), answer in zip(built, answers):
                    f.write(json.dumps({"id": qid, "response": answer}) + "\n")
                f.flush()
        if stopped:
            break
    if stopped:
        break
run_hours = (time.time() - t_start) / 3600
print(f"{'⚠ Stopped at the time limit' if stopped else '✓ All questions answered'} after {run_hours:.1f} h; "
      f"unreadable images: {dict(unreadable) or 'none'}")

DTR plain:   0%|          | 0/53 [00:00<?, ?it/s]

DTR facts:   0%|          | 0/53 [00:00<?, ?it/s]

BBR plain:   0%|          | 0/296 [00:00<?, ?it/s]

BBR facts:   0%|          | 0/296 [00:00<?, ?it/s]

DRE plain:   0%|          | 0/273 [00:00<?, ?it/s]

DRE facts:   0%|          | 0/273 [00:00<?, ?it/s]

DSR plain:   0%|          | 0/251 [00:00<?, ?it/s]

DSR facts:   0%|          | 0/251 [00:00<?, ?it/s]

ORR plain:   0%|          | 0/128 [00:00<?, ?it/s]

✓ All questions answered after 4.3 h; unreadable images: {'BBR': 4, 'DRE': 4, 'DSR': 2}


In [6]:
# ── Cell 6: Score both variants, the six-task KR2 table, the gate, and the output files ──
answers = {}
for variant in VARIANTS:
    for path in (OUT_DIR / "responses" / variant).glob("*.jsonl"):
        for line in open(path, encoding="utf-8"):
            r = json.loads(line)
            answers[(variant, r["id"])] = r["response"]

rows = []
for code, _, _, _ in TASKS:
    for qid, e in questions[code]:
        row = {"task": code, "id": qid, "sensor": e.get("post_image_type") or e.get("image_type") or "Optical",
               "truth": e["ground_truth_option"]}
        for variant in VARIANTS:
            if (variant, qid) in answers:
                row[variant] = answers[(variant, qid)]
                row[f"{variant}_correct"] = letters(answers[(variant, qid)]) == letters(e["ground_truth_option"])
        rows.append(row)
per_q = pd.DataFrame(rows)
per_q.to_csv(OUT_DIR / "H1_v2_per_question.csv", index=False)

def accuracy(df, variant):
    """(accuracy % over the answered questions, how many were answered)."""
    col = f"{variant}_correct"
    answered = df[col].dropna() if col in df else pd.Series(dtype=bool)
    return (round(100 * answered.astype(bool).mean(), 2) if len(answered) else float("nan")), len(answered)

table = []
for code, _, _, with_facts in TASKS:
    df = per_q[per_q["task"] == code]
    (plain, n_plain), (facts, n_facts) = accuracy(df, "plain"), accuracy(df, "facts")
    table.append({"task": code, "questions": len(df), "plain_%": plain, "facts_%": facts,
                  "facts_minus_plain": round(facts - plain, 2) if with_facts else float("nan"),
                  "complete": n_plain == len(df) and (n_facts == len(df) or not with_facts),
                  "note": "" if with_facts else "plain only: crop images without U-Net facts"})
v1_bdc = (V1_SUMMARY.loc[V1_SUMMARY["group"] == "all", "accuracy_%"].item()
          if V1_SUMMARY is not None else float("nan"))
table.insert(2, {"task": "BDC", "questions": 4982, "plain_%": float("nan"), "facts_%": float("nan"),
                 "facts_minus_plain": float("nan"), "complete": V1_SUMMARY is not None,
                 "note": "answered by H1 V1 (U-Net counting), not by the VLM"})
kr2 = pd.DataFrame(table)

# Gate: keep the facts only if no task drops by more than 1 point against the plain run
fact_rows = kr2[kr2["task"].isin([c for c, _, _, with_facts in TASKS if with_facts])]
worst = float(fact_rows["facts_minus_plain"].min())
keep_facts = bool(worst >= -1.0) and bool(fact_rows["complete"].all())
kr2["kr2_system_%"] = [v1_bdc if t == "BDC" else (f if keep_facts and not pd.isna(f) else p)
                       for t, p, f in zip(kr2["task"], kr2["plain_%"], kr2["facts_%"])]
kr2["ref_499A_finetuned_%"] = kr2["task"].map(REF_FT)
kr2["ref_base_%"] = kr2["task"].map(REF_BASE)
kr2.to_csv(OUT_DIR / "kr2_results.csv", index=False)
print(kr2.drop(columns=["note"]).to_string(index=False))

by_sensor = (per_q.groupby(["task", "sensor"])
             .agg(questions=("id", "size"),
                  plain_pct=("plain_correct", lambda s: round(100 * s.dropna().astype(bool).mean(), 2)),
                  facts_pct=("facts_correct", lambda s: round(100 * s.dropna().astype(bool).mean(), 2))
                  if "facts_correct" in per_q else ("id", lambda s: float("nan")))
             .reset_index())
print("\nBy sensor of the after-image:")
print(by_sensor.to_string(index=False))

gate = (f"Facts {'kept' if keep_facts else 'not kept'}: the largest change against the plain run is {worst:+.2f} "
        f"points (rule: no task may drop by more than 1 point"
        f"{'; some task is incomplete' if not fact_rows['complete'].all() else ''}).")
kr2_means = {c: round(float(kr2[c].mean()), 2) for c in ("kr2_system_%", "ref_499A_finetuned_%", "ref_base_%")}
print(f"\n{gate}")
print(f"Six-task mean: KR2 system {kr2_means['kr2_system_%']}%, 499A fine-tuned {kr2_means['ref_499A_finetuned_%']}%, "
      f"base {kr2_means['ref_base_%']}% (BDC: V1 {v1_bdc}% against the base model's {REF_BASE['BDC']}%)")

# The answers in the 499A format, so 4_score_results.py can score them too
for variant in VARIANTS:
    tag = f"{MODEL_ID.replace('/', '--')}-greedy" + ("-facts" if variant == "facts" else "")
    for path in (OUT_DIR / "responses" / variant).glob("*.jsonl"):
        dst = OUT_DIR / "results" / path.stem / tag / "finished.jsonl"
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(path, dst)

def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

config = {"task": "H1 V2", "smoke_test": SMOKE_TEST, "model": MODEL_ID, "decoding": "greedy",
          "image_size": IMAGE_SIZE, "batch": BATCH, "max_new_tokens": MAX_NEW_TOKENS, "transformers": transformers.__version__,
          "torch": torch.__version__, "stopped_at_time_limit": stopped, "run_hours": round(run_hours, 2),
          "unreadable_images": dict(unreadable), "keep_facts": keep_facts, "gate": gate, "six_task_means": kr2_means}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join([
    "# H1 V2: counts as text, all six tasks", "",
    f"Model `{MODEL_ID}`, greedy decoding, images {IMAGE_SIZE} x {IMAGE_SIZE}, official DisasterM3 prompts"
    f"{' (SMOKE TEST)' if SMOKE_TEST else ''}. `plain` is the official prompt; `facts` adds H1 V1's fact line "
    "before the question. BDC comes from H1 V1 (U-Net counting).", "",
    "## Six-task table", "", as_markdown(kr2), "",
    f"**Gate:** {gate}", "",
    f"Six-task mean: KR2 system {kr2_means['kr2_system_%']}%, 499A fine-tuned 7B {kr2_means['ref_499A_finetuned_%']}%, "
    f"base model {kr2_means['ref_base_%']}%.", "",
    "## By sensor of the after-image", "", as_markdown(by_sensor), "",
    "## Notes", "",
    "- The 499A references were most likely sampled at temperature 1.0 (a re-run gave Disaster Type 64.05% "
    "instead of 68.33%), so compare `facts` with this run's `plain`, not with the references.",
    "- SAR questions get the fact line of the optical twin of the same tile.", "",
    "## Files", "",
    "- `kr2_results.csv`, `H1_v2_per_question.csv`, `config.json`",
    "- `responses/<variant>/<subset>.jsonl` and `results/<subset>/<tag>/finished.jsonl` (for `4_score_results.py`)",
]), encoding="utf-8")
print(f"✓ Wrote kr2_results.csv, H1_v2_per_question.csv, results/, README.md and config.json to {OUT_DIR}")

task  questions  plain_%  facts_%  facts_minus_plain  complete  kr2_system_%  ref_499A_finetuned_%  ref_base_%
 DTR        420    70.71    69.29              -1.42      True         70.71                 68.33        66.6
 BBR       2363    20.52    10.66              -9.86      True         20.52                 16.39         4.7
 BDC       4982      NaN      NaN                NaN      True         22.26                 23.58        34.2
 DRE       2178    31.86    30.07              -1.79      True         31.86                 26.24        29.3
 DSR       2007    37.87    36.42              -1.45      True         37.87                 31.31        28.3
 ORR       1018    22.10      NaN                NaN      True         22.10                 22.50        23.9

By sensor of the after-image:
task  sensor  questions  plain_pct  facts_pct
 BBR Optical       2006      19.54      10.17
 BBR     SAR        357      26.05      13.45
 DRE Optical       1868      32.87      31.00
 DRE    